# Daily Challenge — MCP + Airbnb Mini-Agent

**Week 10 · Day 2 — Bootcamp GenAI & Machine Learning 2026**

A mini MCP agent that combines:
- **Notes MCP server** (local FastMCP) — `add_note`, `list_notes`
- **Airbnb MCP server** (stub or real via `npx`) — `airbnb_search`
- **Stub/real LLM planner** — decides which tool to call

```
User prompt
    └──► Planner (stub/LLM)  ←── tools from BOTH servers
              │ tool_calls
              ▼
         Router ──► NotesServer  (STDIO)
                └──► AirbnbServer (STDIO / npm)
```

---
## Step 1 — Install

In [ ]:
%pip install -q "mcp[cli]" openai nest_asyncio

# Optional: install real Airbnb MCP server (requires npm)
# !npm install -g @openbnb/mcp-server-airbnb

---
## Step 2 — Config

In [ ]:
import os
import nest_asyncio
nest_asyncio.apply()  # allow asyncio in Jupyter

# --- Toggle real vs stub ---
USE_REAL_AIRBNB = False   # Set True + npm install for real Airbnb MCP
USE_REAL_LLM    = False   # Set True + GITHUB_TOKEN for real LLM
GITHUB_TOKEN    = os.environ.get("GITHUB_TOKEN", "")

print(f"Airbnb : {'real (npm)' if USE_REAL_AIRBNB else 'stub'}")
print(f"LLM    : {'real (GitHub Models)' if USE_REAL_LLM and GITHUB_TOKEN else 'stub'}")

---
## Step 3 — Notes MCP Server

In [ ]:
%%writefile notes_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("NotesServer")
_notes = []


@mcp.tool()
def add_note(title: str, content: str) -> str:
    """Add a note with a title and content. Returns confirmation."""
    _notes.append({"title": title, "content": content})
    return f"Note '{title}' saved. Total notes: {len(_notes)}"


@mcp.tool()
def list_notes() -> str:
    """List all saved notes."""
    if not _notes:
        return "No notes yet."
    return "\n".join(f"• {n['title']}: {n['content']}" for n in _notes)


if __name__ == "__main__":
    mcp.run()

---
## Step 4 — Airbnb Stub Server

In [ ]:
%%writefile airbnb_stub_server.py
import json
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("AirbnbStub")

LISTINGS = {
    "paris": [
        {"id": "P001", "name": "Cozy Studio near Eiffel Tower", "price_night": 85,  "rating": 4.8},
        {"id": "P002", "name": "Montmartre Artist Loft",         "price_night": 110, "rating": 4.6},
        {"id": "P003", "name": "Modern Flat in Le Marais",        "price_night": 130, "rating": 4.9},
    ],
    "london": [
        {"id": "L001", "name": "Shoreditch Brick Lane Apartment", "price_night": 95,  "rating": 4.7},
        {"id": "L002", "name": "Covent Garden Studio",            "price_night": 120, "rating": 4.5},
    ],
    "nyc": [
        {"id": "N001", "name": "Brooklyn Heights Studio",  "price_night": 150, "rating": 4.8},
        {"id": "N002", "name": "Manhattan Midtown Room",   "price_night": 200, "rating": 4.4},
    ],
}


@mcp.tool()
def airbnb_search(city: str, max_price: int = 500) -> str:
    """Search Airbnb listings for a city. Optionally filter by max price per night."""
    results = LISTINGS.get(city.lower(), [])
    filtered = [l for l in results if l["price_night"] <= max_price]
    if not filtered:
        return json.dumps({"listings": [], "city": city,
                           "message": f"No listings under ${max_price}/night."})
    return json.dumps({"city": city, "count": len(filtered),
                       "listings": filtered}, indent=2)


if __name__ == "__main__":
    mcp.run()

---
## Step 5 — Client Helpers

In [ ]:
import asyncio
import json
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Server parameters
notes_params = StdioServerParameters(command="mcp", args=["run", "notes_server.py"], env=None)
airbnb_params = (
    StdioServerParameters(command="npx", args=["-y", "@openbnb/mcp-server-airbnb"], env=None)
    if USE_REAL_AIRBNB
    else StdioServerParameters(command="mcp", args=["run", "airbnb_stub_server.py"], env=None)
)


def convert_to_llm_tool(mcp_tool) -> dict:
    """Convert MCP tool → OpenAI function spec."""
    return {
        "type": "function",
        "function": {
            "name": mcp_tool.name,
            "description": mcp_tool.description or "",
            "parameters": mcp_tool.inputSchema,
        }
    }


async def get_tools(params) -> list:
    """Connect to a server and return its tools."""
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            result = await s.list_tools()
            return result.tools


async def call_tool(params, tool_name: str, args: dict) -> str:
    """Connect to a server, call a tool, return result text."""
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            result = await s.call_tool(tool_name, args)
            return result.content[0].text


print("Client helpers defined ✅")

---
## Step 6 — Planner (Stub or Real LLM)

In [ ]:
import re


def stub_planner(prompt: str, llm_tools: list) -> list:
    """Simulate an LLM: parse prompt keywords → return tool_calls."""
    p = prompt.lower()

    tool_calls = []

    # Airbnb search
    if any(k in p for k in ["airbnb", "listing", "search", "find", "rent", "stay"]):
        # Extract city name (simple heuristic)
        cities = ["paris", "london", "nyc", "new york"]
        city = next((c for c in cities if c in p), "paris")
        city = "nyc" if city == "new york" else city
        # Extract max price if mentioned
        prices = re.findall(r'\$?(\d+)', prompt)
        max_price = int(prices[0]) if prices else 500
        tool_calls.append({"name": "airbnb_search",
                           "arguments": {"city": city, "max_price": max_price}})

    # Add note
    if any(k in p for k in ["note", "save", "remember", "write down"]):
        # Extract title from prompt (after "note:", "save", etc.)
        title_match = re.search(r'note[:\s]+([\w\s]+)', p)
        title = title_match.group(1).strip().title() if title_match else "Travel Note"
        tool_calls.append({"name": "add_note",
                           "arguments": {"title": title, "content": prompt}})

    # List notes
    if any(k in p for k in ["list notes", "show notes", "my notes"]):
        tool_calls.append({"name": "list_notes", "arguments": {}})

    return tool_calls


def real_llm_planner(prompt: str, llm_tools: list) -> list:
    from openai import OpenAI
    client = OpenAI(
        base_url="https://models.inference.ai.azure.com",
        api_key=GITHUB_TOKEN
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        tools=llm_tools,
        tool_choice="auto"
    )
    msg = response.choices[0].message
    if msg.tool_calls:
        return [
            {"name": tc.function.name,
             "arguments": json.loads(tc.function.arguments)}
            for tc in msg.tool_calls
        ]
    return []


def planner(prompt: str, llm_tools: list) -> list:
    if USE_REAL_LLM and GITHUB_TOKEN:
        return real_llm_planner(prompt, llm_tools)
    return stub_planner(prompt, llm_tools)


# Tool → server routing map
NOTES_TOOLS   = {"add_note", "list_notes"}
AIRBNB_TOOLS  = {"airbnb_search"}

print("Planner defined ✅")

---
## Step 7 — Orchestrator

In [ ]:
async def orchestrate(prompt: str):
    print(f"\n{'='*55}")
    print(f"Prompt: {prompt!r}")
    print(f"{'='*55}\n")

    # 1. Collect tools from both servers
    notes_tools  = await get_tools(notes_params)
    airbnb_tools = await get_tools(airbnb_params)
    all_tools    = notes_tools + airbnb_tools
    llm_tools    = [convert_to_llm_tool(t) for t in all_tools]

    print("Available tools:", [t["function"]["name"] for t in llm_tools])
    print()

    # 2. Plan
    tool_calls = planner(prompt, llm_tools)
    print("Planned tool calls:")
    for tc in tool_calls:
        print(f"  → {tc['name']}({tc['arguments']})")
    print()

    # 3. Execute — route to correct server
    print("Results:")
    for tc in tool_calls:
        name = tc["name"]
        args = tc["arguments"]

        if name in NOTES_TOOLS:
            server = "NotesServer"
            result = await call_tool(notes_params, name, args)
        elif name in AIRBNB_TOOLS:
            server = "AirbnbStub" if not USE_REAL_AIRBNB else "AirbnbReal"
            result = await call_tool(airbnb_params, name, args)
        else:
            result = f"Unknown tool: {name}"
            server = "?"

        print(f"  [{server}] {name}:")
        # Pretty-print JSON results
        try:
            parsed = json.loads(result)
            print(json.dumps(parsed, indent=4))
        except Exception:
            print(f"  {result}")
        print()

print("Orchestrator defined ✅")

---
## Step 8 — Demo Run

> **TODO:** Tweak the prompts below to test different cities or notes.

In [ ]:
# Demo 1: Search Airbnb listings in Paris under $120/night
asyncio.run(orchestrate("Search Airbnb listings in Paris under $120/night."))

In [ ]:
# Demo 2: Add a travel note
asyncio.run(orchestrate("Save a note: Paris Trip — Book accommodation near Eiffel Tower."))

In [ ]:
# Demo 3: Combined — search + note in one prompt
asyncio.run(orchestrate("Find Airbnb listings in London and save a note: London research done."))

In [ ]:
# Demo 4: List all saved notes
asyncio.run(orchestrate("Show my notes."))

---
## Expected Output

```
=======================================================
Prompt: 'Search Airbnb listings in Paris under $120/night.'
=======================================================

Available tools: ['add_note', 'list_notes', 'airbnb_search']

Planned tool calls:
  → airbnb_search({'city': 'paris', 'max_price': 120})

Results:
  [AirbnbStub] airbnb_search:
    {
        "city": "paris",
        "count": 2,
        "listings": [
            {"id": "P001", "name": "Cozy Studio near Eiffel Tower", "price_night": 85, "rating": 4.8},
            {"id": "P002", "name": "Montmartre Artist Loft", "price_night": 110, "rating": 4.6}
        ]
    }
```

---
## Architecture Summary

| Component | Role |
|-----------|------|
| `notes_server.py` | Local FastMCP — `add_note`, `list_notes` |
| `airbnb_stub_server.py` | Stub Airbnb FastMCP — `airbnb_search` with fixed data |
| `convert_to_llm_tool` | MCP tool → OpenAI function spec |
| `stub_planner` | Keyword-based tool selector (no LLM needed) |
| `orchestrate` | Collects tools → plans → routes → executes |

**Key pattern:** The agent doesn't know in advance which server owns which tool. It collects all tools, lets the planner decide, then routes each `tool_call` to the right server by name.